# Chapter 7: Four Valuation Yields
In the lecture, we calculated four valuation yields for Empire. We will now follow the same steps for our sample of 50 firms. Each yield tells us how much earnings, operating profit, free cash flow, or gross profit we receive relative to the price of the company.

Open this notebook in Google Colab and choose **Runtime → Run all**, or run one cell at a time. Read the short explanation before each calculation. The code is supplied, and the exercise needs no credentials or private files.


## 1. Load the data
We use the same public CSV as the earlier chapters. `df` is our pandas table. First, display a few rows so we can see the company names, fiscal years, and financial information.


In [1]:
import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/x2003xuy/acct3343-public/main/data/acct3343_financial_data_50.csv'
df = pd.read_csv(DATA_URL)
display(df.head())


,ticker,fiscal_period_end,fiscal_year,company_name,sector,industry,exchange,currency,current_share_price,current_market_cap,...,issuance_of_capital_stock,repurchase_of_capital_stock,net_common_stock_issuance,stock_based_compensation,cash_dividends_paid,issuance_of_debt,repayment_of_debt,fiscal_price_date,fiscal_period_end_close,fiscal_period_end_adjusted_close
0,AAPL,2025-09-30,2025,Apple Inc.,Technology,Consumer Electronics,NMS,USD,331.34,4835635625984,...,NaN,-9.071100e+10,-9.071100e+10,1.286300e+10,-1.542100e+10,4.481000e+09,-1.093200e+10,2025-09-30,254.630005,253.692871
1,AAPL,2024-09-30,2024,Apple Inc.,Technology,Consumer Electronics,NMS,USD,331.34,4835635625984,...,NaN,-9.494900e+10,-9.494900e+10,1.168800e+10,-1.523400e+10,0.000000e+00,-9.958000e+09,2024-09-30,233.000000,231.067093
2,AAPL,2023-09-30,2023,Apple Inc.,Technology,Consumer Electronics,NMS,USD,331.34,4835635625984,...,NaN,-7.755000e+10,-7.755000e+10,1.083300e+10,-1.502500e+10,5.228000e+09,-1.115100e+10,2023-09-29,171.210007,168.925262
3,AAPL,2022-09-30,2022,Apple Inc.,Technology,Consumer Electronics,NMS,USD,331.34,4835635625984,...,NaN,-8.940200e+10,-8.940200e+10,9.038000e+09,-1.484100e+10,5.465000e+09,-9.543000e+09,2022-09-30,138.199997,135.551590
4,AAPL,2021-09-30,2021,Apple Inc.,Technology,Consumer Electronics,NMS,USD,331.34,4835635625984,...,1.105000e+09,NaN,NaN,NaN,NaN,NaN,NaN,2021-09-30,141.500000,138.015015


## 2. Keep the latest fiscal year for each firm
Convert the fiscal date to a date, sort each company's observations from oldest to newest, and keep its last row. We keep the latest year even when one of its financial fields is missing.

The statement figures and `current_market_cap` are saved observations from the course dataset. Market capitalization is a saved quote snapshot, rather than a historical value on each fiscal date. The 50 firms in this file use USD amounts in full currency units; do not mix dollars with millions. These are descriptive classroom comparisons.


In [2]:
df["fiscal_period_end"] = pd.to_datetime(df["fiscal_period_end"])
df = df.sort_values(["ticker", "fiscal_period_end"])
df = df.drop_duplicates("ticker", keep="last").copy()

print("Firms:", len(df))
display(df[["ticker", "company_name", "fiscal_period_end", "currency"]].head(10))


Firms: 50


,ticker,company_name,fiscal_period_end,currency
0,AAPL,Apple Inc.,2025-09-30,USD
5,ABBV,AbbVie Inc.,2025-12-31,USD
10,ABT,Abbott Laboratories,2025-12-31,USD
15,ADBE,Adobe Inc.,2025-11-30,USD
20,AEP,"American Electric Power Company, Inc.",2025-12-31,USD
25,AMZN,"Amazon.com, Inc.",2025-12-31,USD
30,BAC,Bank of America Corporation,2025-12-31,USD
35,BLK,"BlackRock, Inc.",2025-12-31,USD
39,CAT,Caterpillar Inc.,2025-12-31,USD
44,COP,ConocoPhillips,2025-12-31,USD


## 3. Calculate Earnings Yield
**Earnings Yield = Net Income / Market Capitalization.**

Use `net_income` and `current_market_cap`. For example, a yield of `0.05` means earnings are 5% of the firm's market value. `.where(... > 0)` allows division only when market capitalization is positive. A missing input produces a missing yield.


In [3]:
df["Earnings_Yield"] = df["net_income"] / df["current_market_cap"].where(df["current_market_cap"] > 0)
display(df[["ticker", "net_income", "current_market_cap", "Earnings_Yield"]].head(10))


,ticker,net_income,current_market_cap,Earnings_Yield
0,AAPL,1.120100e+11,4835635625984,0.023163
5,ABBV,4.226000e+09,464760700928,0.009093
10,ABT,6.524000e+09,176551018496,0.036952
15,ADBE,7.130000e+09,102459604992,0.069588
20,AEP,3.580000e+09,65659768832,0.054523
25,AMZN,7.767000e+10,2679535828992,0.028986
30,BAC,3.050900e+10,415998607360,0.073339
35,BLK,5.553000e+09,171904696320,0.032303
39,CAT,8.884000e+09,359874887680,0.024686
44,COP,7.988000e+09,169664888832,0.047081


## 4. Calculate Total Enterprise Value
**TEV = Market Capitalization + Total Debt − Cash.**

The revised lecture includes current debt, long-term debt, and lease obligations. The CSV already combines debt and leases in two columns:

- `current_debt_and_capital_lease_obligation`
- `long_term_debt_and_capital_lease_obligation`

Add these two columns once to obtain `Total_Debt`. Then subtract `cash_and_cash_equivalents`. Adding `total_debt`, plain debt, or leases again would double count. If either required debt component is missing, total debt and TEV remain missing; a missing debt figure does not mean zero.


In [4]:
df["Total_Debt"] = df["current_debt_and_capital_lease_obligation"] + df["long_term_debt_and_capital_lease_obligation"]
df["TEV"] = df["current_market_cap"].where(df["current_market_cap"] > 0) + df["Total_Debt"] - df["cash_and_cash_equivalents"]

display(df[["ticker", "current_market_cap", "current_debt_and_capital_lease_obligation", "long_term_debt_and_capital_lease_obligation", "Total_Debt", "cash_and_cash_equivalents", "TEV"]].head(10))


,ticker,current_market_cap,current_debt_and_capital_lease_obligation,long_term_debt_and_capital_lease_obligation,Total_Debt,cash_and_cash_equivalents,TEV
0,AAPL,4835635625984,2.032900e+10,7.832800e+10,9.865700e+10,3.593400e+10,4.898359e+12
5,ABBV,464760700928,8.555000e+09,5.894100e+10,6.749600e+10,5.229000e+09,5.270277e+11
10,ABT,176551018496,3.033000e+09,1.082700e+10,1.386000e+10,8.522000e+09,1.818890e+11
15,ADBE,102459604992,7.700000e+07,6.571000e+09,6.648000e+09,5.431000e+09,1.036766e+11
20,AEP,65659768832,4.802000e+09,4.470600e+10,4.950800e+10,1.970000e+08,1.149708e+11
25,AMZN,2679535828992,NaN,1.529870e+11,NaN,8.681000e+10,NaN
30,BAC,415998607360,4.808800e+10,3.178160e+11,3.659040e+11,2.393190e+11,5.425836e+11
35,BLK,171904696320,NaN,1.499600e+10,NaN,1.146800e+10,NaN
39,CAT,359874887680,1.263400e+10,3.069600e+10,4.333000e+10,9.980000e+09,3.932249e+11
44,COP,169664888832,1.020000e+09,2.242400e+10,2.344400e+10,6.497000e+09,1.866119e+11


## 5. Calculate Enterprise Yield
**Enterprise Yield = EBIT / TEV.**

Use the supplied `ebit` column. We divide only by positive TEV. If TEV is zero, negative, or missing, this yield is unavailable. We apply the same rule to the next two yields.


In [5]:
df["Enterprise_Yield"] = df["ebit"] / df["TEV"].where(df["TEV"] > 0)
display(df[["ticker", "ebit", "TEV", "Enterprise_Yield"]].head(10))


,ticker,ebit,TEV,Enterprise_Yield
0,AAPL,1.330500e+11,4.898359e+12,0.027162
5,ABBV,9.490000e+09,5.270277e+11,0.018007
10,ABT,8.959000e+09,1.818890e+11,0.049255
15,ADBE,8.997000e+09,1.036766e+11,0.086779
20,AEP,5.750000e+09,1.149708e+11,0.050013
25,AMZN,9.958500e+10,NaN,NaN
30,BAC,NaN,5.425836e+11,NaN
35,BLK,8.233000e+09,NaN,NaN
39,CAT,1.204300e+10,3.932249e+11,0.030626
44,COP,1.388900e+10,1.866119e+11,0.074427


## 6. Calculate Free Cash Flow Yield
First calculate **FCF = Cash Flow from Operations − Capital Expenditures**.

Use `operating_cash_flow` and `capital_expenditure`. The CSV records CapEx spending as a negative cash outflow. `.abs()` expresses spending as a positive amount, allowing us to subtract it under either a negative-outflow or positive-spending convention. A missing CapEx value stays missing.


In [6]:
df["CapEx_Spending"] = df["capital_expenditure"].abs()
df["FCF"] = df["operating_cash_flow"] - df["CapEx_Spending"]
display(df[["ticker", "operating_cash_flow", "capital_expenditure", "CapEx_Spending", "FCF"]].head(10))


,ticker,operating_cash_flow,capital_expenditure,CapEx_Spending,FCF
0,AAPL,1.114820e+11,-1.271500e+10,1.271500e+10,9.876700e+10
5,ABBV,1.903000e+10,-1.214000e+09,1.214000e+09,1.781600e+10
10,ABT,9.566000e+09,-2.171000e+09,2.171000e+09,7.395000e+09
15,ADBE,1.003100e+10,-1.790000e+08,1.790000e+08,9.852000e+09
20,AEP,6.944000e+09,-8.583000e+09,8.583000e+09,-1.639000e+09
25,AMZN,1.395140e+11,-1.318190e+11,1.318190e+11,7.695000e+09
30,BAC,1.261300e+10,NaN,NaN,NaN
35,BLK,3.927000e+09,-3.750000e+08,3.750000e+08,3.552000e+09
39,CAT,1.173900e+10,-4.286000e+09,4.286000e+09,7.453000e+09
44,COP,1.979600e+10,-1.255300e+10,1.255300e+10,7.243000e+09


Now divide free cash flow by TEV: **FCF Yield = FCF / TEV**. Negative free cash flow is a valid result and produces a negative yield when TEV is positive.


In [7]:
df["FCF_Yield"] = df["FCF"] / df["TEV"].where(df["TEV"] > 0)
display(df[["ticker", "FCF", "TEV", "FCF_Yield"]].head(10))


,ticker,FCF,TEV,FCF_Yield
0,AAPL,9.876700e+10,4.898359e+12,0.020163
5,ABBV,1.781600e+10,5.270277e+11,0.033805
10,ABT,7.395000e+09,1.818890e+11,0.040657
15,ADBE,9.852000e+09,1.036766e+11,0.095026
20,AEP,-1.639000e+09,1.149708e+11,-0.014256
25,AMZN,7.695000e+09,NaN,NaN
30,BAC,NaN,5.425836e+11,NaN
35,BLK,3.552000e+09,NaN,NaN
39,CAT,7.453000e+09,3.932249e+11,0.018954
44,COP,7.243000e+09,1.866119e+11,0.038813


## 7. Calculate Gross Profits Yield
First calculate **Gross Profit = Revenue − Cost of Goods Sold**. Our revenue column is `total_revenue`; our COGS column is `cost_of_revenue`. Gross profit is an amount, so keep it in the same currency and units as TEV.


In [8]:
df["Gross_Profit"] = df["total_revenue"] - df["cost_of_revenue"]
display(df[["ticker", "total_revenue", "cost_of_revenue", "Gross_Profit"]].head(10))


,ticker,total_revenue,cost_of_revenue,Gross_Profit
0,AAPL,4.161610e+11,2.209600e+11,1.952010e+11
5,ABBV,6.116000e+10,1.820400e+10,4.295600e+10
10,ABT,4.432800e+10,1.931900e+10,2.500900e+10
15,ADBE,2.376900e+10,2.551000e+09,2.121800e+10
20,AEP,2.187600e+10,8.530000e+09,1.334600e+10
25,AMZN,7.169240e+11,3.564140e+11,3.605100e+11
30,BAC,1.130970e+11,NaN,NaN
35,BLK,2.421600e+10,1.290600e+10,1.131000e+10
39,CAT,6.758900e+10,4.611100e+10,2.147800e+10
44,COP,5.894400e+10,4.415600e+10,1.478800e+10


Now calculate **Gross Profits Yield = Gross Profit / TEV**. This uses gross profit dollars, rather than the gross margin percentage.


In [9]:
df["Gross_Profits_Yield"] = df["Gross_Profit"] / df["TEV"].where(df["TEV"] > 0)
display(df[["ticker", "Gross_Profit", "TEV", "Gross_Profits_Yield"]].head(10))


,ticker,Gross_Profit,TEV,Gross_Profits_Yield
0,AAPL,1.952010e+11,4.898359e+12,0.039850
5,ABBV,4.295600e+10,5.270277e+11,0.081506
10,ABT,2.500900e+10,1.818890e+11,0.137496
15,ADBE,2.121800e+10,1.036766e+11,0.204656
20,AEP,1.334600e+10,1.149708e+11,0.116082
25,AMZN,3.605100e+11,NaN,NaN
30,BAC,NaN,5.425836e+11,NaN
35,BLK,1.131000e+10,NaN,NaN
39,CAT,2.147800e+10,3.932249e+11,0.054620
44,COP,1.478800e+10,1.866119e+11,0.079245


## 8. Compare and rank the firms
Bring the four yields together in one table. The display below shows percentages: `0.05` appears as `5.00%`. Formatting changes how the results look; the underlying calculations retain full precision. **Unavailable** means a required input or positive denominator is missing.


In [10]:
yield_columns = ["Earnings_Yield", "Enterprise_Yield", "FCF_Yield", "Gross_Profits_Yield"]
summary = df[["ticker", "company_name", "fiscal_period_end"] + yield_columns].copy()
display(summary.style.format("{:.2%}", subset=yield_columns, na_rep="Unavailable"))


,ticker,company_name,fiscal_period_end,Earnings_Yield,Enterprise_Yield,FCF_Yield,Gross_Profits_Yield
0,AAPL,Apple Inc.,2025-09-30 00:00:00,2.32%,2.72%,2.02%,3.99%
5,ABBV,AbbVie Inc.,2025-12-31 00:00:00,0.91%,1.80%,3.38%,8.15%
10,ABT,Abbott Laboratories,2025-12-31 00:00:00,3.70%,4.93%,4.07%,13.75%
15,ADBE,Adobe Inc.,2025-11-30 00:00:00,6.96%,8.68%,9.50%,20.47%
20,AEP,"American Electric Power Company, Inc.",2025-12-31 00:00:00,5.45%,5.00%,-1.43%,11.61%
25,AMZN,"Amazon.com, Inc.",2025-12-31 00:00:00,2.90%,Unavailable,Unavailable,Unavailable
30,BAC,Bank of America Corporation,2025-12-31 00:00:00,7.33%,Unavailable,Unavailable,Unavailable
35,BLK,"BlackRock, Inc.",2025-12-31 00:00:00,3.23%,Unavailable,Unavailable,Unavailable
39,CAT,Caterpillar Inc.,2025-12-31 00:00:00,2.47%,3.06%,1.90%,5.46%
44,COP,ConocoPhillips,2025-12-31 00:00:00,4.71%,7.44%,3.88%,7.92%


### Earnings Yield: highest to lowest
A higher yield ranks first. `ascending=False` puts the largest yield at the top; `method="min"` gives equal yields the same rank. Missing yields receive no rank and sort last. We calculate ranks using unrounded values.

The next four tables show the top ten firms for each measure. Remove `.head(10)` to display all firms.


In [11]:
df["Earnings_Yield_Rank"] = df["Earnings_Yield"].rank(ascending=False, method="min").astype("Int64")
display(df[["ticker", "Earnings_Yield", "Earnings_Yield_Rank"]].sort_values("Earnings_Yield", ascending=False, na_position="last").head(10))


,ticker,Earnings_Yield,Earnings_Yield_Rank
233,WFC,0.078647,1
98,HON,0.073343,2
30,BAC,0.073339,3
15,ADBE,0.069588,4
228,UPS,0.063970,5
78,EOG,0.061735,6
123,LOW,0.060957,7
113,JPM,0.060690,8
88,GS,0.060408,9
133,MO,0.059598,10


### Enterprise Yield: highest to lowest
Rank EBIT relative to the price of the whole business.


In [12]:
df["Enterprise_Yield_Rank"] = df["Enterprise_Yield"].rank(ascending=False, method="min").astype("Int64")
display(df[["ticker", "Enterprise_Yield", "Enterprise_Yield_Rank"]].sort_values("Enterprise_Yield", ascending=False, na_position="last").head(10))


,ticker,Enterprise_Yield,Enterprise_Yield_Rank
15,ADBE,0.086779,1
98,HON,0.077895,2
78,EOG,0.076996,3
133,MO,0.076676,4
228,UPS,0.074504,5
44,COP,0.074427,6
123,LOW,0.067215,7
158,NKE,0.066387,8
183,PG,0.058114,9
138,MRK,0.057567,10


### Free Cash Flow Yield: highest to lowest
Rank the cash remaining after capital investment relative to TEV.


In [13]:
df["FCF_Yield_Rank"] = df["FCF_Yield"].rank(ascending=False, method="min").astype("Int64")
display(df[["ticker", "FCF_Yield", "FCF_Yield_Rank"]].sort_values("FCF_Yield", ascending=False, na_position="last").head(10))


,ticker,FCF_Yield,FCF_Yield_Rank
15,ADBE,0.095026,1
133,MO,0.065849,2
53,CRM,0.065346,3
98,HON,0.061927,4
198,SLB,0.051039,5
123,LOW,0.050054,6
228,UPS,0.043395,7
183,PG,0.041416,8
178,PFE,0.041290,9
218,UNH,0.041064,10


### Gross Profits Yield: highest to lowest
Rank gross profit relative to TEV. Each measure ranks firms separately; they can give different answers.


In [14]:
df["Gross_Profits_Yield_Rank"] = df["Gross_Profits_Yield"].rank(ascending=False, method="min").astype("Int64")
display(df[["ticker", "Gross_Profits_Yield", "Gross_Profits_Yield_Rank"]].sort_values("Gross_Profits_Yield", ascending=False, na_position="last").head(10))


,ticker,Gross_Profits_Yield,Gross_Profits_Yield_Rank
158,NKE,0.348125,1
173,PEP,0.225204,2
218,UNH,0.211812,3
178,PFE,0.211622,4
15,ADBE,0.204656,5
238,WMT,0.193952,6
123,LOW,0.188972,7
78,EOG,0.167041,8
98,HON,0.157948,9
93,HD,0.148768,10


### See which ratios are unavailable
Keep all 50 firms, even if one ratio is missing. For example, a firm can have a valid Earnings Yield but an unavailable Enterprise Yield because EBIT or a debt component is missing. These counts show the number of unavailable results, and the table identifies the firms. Inspect their input columns in the earlier steps.


In [15]:
display(summary[yield_columns].isna().sum().rename("Unavailable firms"))
display(summary[summary[yield_columns].isna().any(axis=1)])


Earnings_Yield         0
Enterprise_Yield       8
FCF_Yield              6
Gross_Profits_Yield    8
Name: Unavailable firms, dtype: int64

,ticker,company_name,fiscal_period_end,Earnings_Yield,Enterprise_Yield,FCF_Yield,Gross_Profits_Yield
25,AMZN,"Amazon.com, Inc.",2025-12-31,0.028986,NaN,NaN,NaN
30,BAC,Bank of America Corporation,2025-12-31,0.073339,NaN,NaN,NaN
35,BLK,"BlackRock, Inc.",2025-12-31,0.032303,NaN,NaN,NaN
49,COST,Costco Wholesale Corporation,2025-08-31,0.020261,NaN,NaN,NaN
88,GS,"The Goldman Sachs Group, Inc.",2025-12-31,0.060408,NaN,-0.093275,NaN
113,JPM,JPMorgan Chase & Co.,2025-12-31,0.060690,NaN,NaN,NaN
143,MS,Morgan Stanley,2025-12-31,0.052051,NaN,-0.033902,NaN
233,WFC,Wells Fargo & Company,2025-12-31,0.078647,NaN,NaN,NaN


## 9. Interpret the results
A higher yield means more of the chosen earnings or cash-flow measure relative to the firm's price, so it appears cheaper **on that measure**. A lower yield means less of that measure relative to price. A negative yield reflects a negative numerator; it does not automatically make the firm attractive.

Find the highest-ranked firm under each measure. Does the same firm lead all four lists? Pick two firms and explain why their rankings differ. Compare similar businesses: financial firms, in particular, may lack comparable EBIT or COGS figures. A high yield is a starting point for investigation, not proof that a stock is a good investment.
